In [1]:
import json
import os
import shutil
from random import randint
from typing import Union


import torch
import torch.nn as nn
import torch.onnx as torch_onnx

import numpy as np
import onnxruntime as ort
import onnx
from onnx import helper, numpy_helper, TensorProto

from ml_runner_exporter.onnx_exporter import export_onnx

In [2]:
fixtures_path = "tests/fixtures"

if os.path.exists(fixtures_path):
    shutil.rmtree(fixtures_path)
os.makedirs(fixtures_path)

In [3]:
def get_output_size() -> int:
    return randint(2, 10) * 5

In [4]:
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size, layer_num):
        super(SimpleLinearModel, self).__init__()
        self.layers = nn.ModuleList()
        if layer_num == 1:
            self.layers.append(nn.Linear(input_size, output_size))
        else:
            inter_output_size = get_output_size()
            self.layers.append(nn.Linear(input_size, inter_output_size))
            inter_input_size = inter_output_size
            for i in range(layer_num - 2):
                inter_output_size = get_output_size()
                self.layers.append(nn.Linear(inter_input_size, inter_output_size))
                inter_input_size = inter_output_size
            self.layers.append(nn.Linear(inter_input_size, output_size))

    def forward(self, x):
        # Pass input through the linear layer
        output = x
        for layer in self.layers:
            output = layer.forward(output)
        return output

In [5]:
# Dense layers with every activation type chained in between (relu, sigmoid,
# tanh, softmax). Linear/identity activation is intentionally excluded since
# it has no corresponding ONNX node to export from.
class ActivationModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(ActivationModel, self).__init__()
        self.linear1 = nn.Linear(input_size, 8)
        self.act1_relu = nn.ReLU()
        self.linear2 = nn.Linear(8, 12)
        self.act2_sigmoid = nn.Sigmoid()
        self.linear3 = nn.Linear(12, 10)
        self.act3_tanh = nn.Tanh()
        self.linear4 = nn.Linear(10, output_size)
        self.act4_softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.linear1(x)
        x = self.act1_relu(x)
        x = self.linear2(x)
        x = self.act2_sigmoid(x)
        x = self.linear3(x)
        x = self.act3_tanh(x)
        x = self.linear4(x)
        x = self.act4_softmax(x)
        return x

In [6]:
# Two conv layers stacked directly, no activation or flatten in between -
# isolates conv-to-conv chaining (D3 -> D3 -> D3) on its own.
class SimpleConvOnlyModel(nn.Module):
    def __init__(self):
        super(SimpleConvOnlyModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=2, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return x

In [7]:
# Conv -> ReLU (D3 activation) -> Flatten, with no dense layer after -
# isolates the D3 -> Flat transition on its own.
class ConvFlattenModel(nn.Module):
    def __init__(self):
        super(ConvFlattenModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.act_relu = nn.ReLU()
        self.flatten = nn.Flatten()

    def forward(self, x):
        x = self.conv1(x)
        x = self.act_relu(x)
        x = self.flatten(x)
        return x

In [8]:
# The full pipeline: Conv2d -> ReLU -> Conv2d -> ReLU -> Flatten -> Linear ->
# Sigmoid -> Linear -> Tanh -> Linear -> Softmax. Exercises every layer type
# and activation together in one model.
class FullConvModel(nn.Module):
    def __init__(self, output_size):
        super(FullConvModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.act1_relu = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.act2_relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(8 * 4 * 4, 20)
        self.act3_sigmoid = nn.Sigmoid()
        self.linear2 = nn.Linear(20, 15)
        self.act4_tanh = nn.Tanh()
        self.linear3 = nn.Linear(15, output_size)
        self.act5_softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.act1_relu(x)
        x = self.conv2(x)
        x = self.act2_relu(x)
        x = self.flatten(x)
        x = self.linear1(x)
        x = self.act3_sigmoid(x)
        x = self.linear2(x)
        x = self.act4_tanh(x)
        x = self.linear3(x)
        x = self.act5_softmax(x)
        return x

In [9]:
def _init_uniform(shape, hidden_size, rng):
    stdv = 1.0 / np.sqrt(hidden_size)
    return rng.uniform(-stdv, stdv, size=shape).astype(np.float32)


def _reorder_gru_gates(w: np.ndarray) -> np.ndarray:
    """PyTorch gate order [r, z, n] -> ONNX gate order [z, r, n]."""
    r, z, n = np.split(w, 3, axis=0)
    return np.concatenate([z, r, n], axis=0)


def _build_rnn_or_gru_model(op_type, weight_ih, weight_hh, bias_ih, bias_hh, seq_len, input_size, hidden_size, return_sequences):
    if op_type == "GRU":
        weight_ih = _reorder_gru_gates(weight_ih)
        weight_hh = _reorder_gru_gates(weight_hh)
        bias_ih = _reorder_gru_gates(bias_ih)
        bias_hh = _reorder_gru_gates(bias_hh)

    W = weight_ih[np.newaxis, :, :]
    R = weight_hh[np.newaxis, :, :]
    B = np.concatenate([bias_ih, bias_hh])[np.newaxis, :]

    initializers = [
        numpy_helper.from_array(W, name="W"),
        numpy_helper.from_array(R, name="R"),
        numpy_helper.from_array(B, name="B"),
    ]

    # No Squeeze node here: keep the RNN/GRU op's native output as-is and
    # use it directly as the graph output. That's exactly what
    # torch.onnx.export() would emit too (same underlying ONNX op), and
    # your exporter's node-type parser only knows RNN/GRU/etc, not Squeeze.
    # The extra num_directions axis on Y is harmless since test_input/
    # test_output get flattened before being written to the fixture.
    y_out = "Y" if return_sequences else ""
    yh_out = "" if return_sequences else "Y_h"
    node_kwargs = {"hidden_size": hidden_size, "name": op_type.lower()}
    if op_type == "RNN":
        node_kwargs["activations"] = ["Tanh"]
    else:
        node_kwargs["linear_before_reset"] = 1  # matches PyTorch's GRU formula

    main_node = helper.make_node(op_type, inputs=["X", "W", "R", "B"], outputs=[y_out, yh_out], **node_kwargs)
    nodes = [main_node]

    if return_sequences:
        out_shape = [seq_len, 1, 1, hidden_size]  # (seq_len, num_directions, batch, hidden)
        out_name = "Y"
    else:
        out_shape = [1, 1, hidden_size]  # (num_directions, batch, hidden)
        out_name = "Y_h"

    graph = helper.make_graph(
        nodes,
        f"{op_type.lower()}_model",
        inputs=[helper.make_tensor_value_info("X", TensorProto.FLOAT, [seq_len, 1, input_size])],
        outputs=[helper.make_tensor_value_info(out_name, TensorProto.FLOAT, out_shape)],
        initializer=initializers,
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    return model


def _random_rnn_weights(input_size, hidden_size, rng):
    return (
        _init_uniform((hidden_size, input_size), hidden_size, rng),
        _init_uniform((hidden_size, hidden_size), hidden_size, rng),
        _init_uniform((hidden_size,), hidden_size, rng),
        _init_uniform((hidden_size,), hidden_size, rng),
    )


def _random_gru_weights(input_size, hidden_size, rng):
    return (
        _init_uniform((3 * hidden_size, input_size), hidden_size, rng),
        _init_uniform((3 * hidden_size, hidden_size), hidden_size, rng),
        _init_uniform((3 * hidden_size,), hidden_size, rng),
        _init_uniform((3 * hidden_size,), hidden_size, rng),
    )


def make_simple_rnn_only_model(input_size, hidden_size, seq_len, seed=None):
    rng = np.random.default_rng(seed)
    weight_ih, weight_hh, bias_ih, bias_hh = _random_rnn_weights(input_size, hidden_size, rng)
    return _build_rnn_or_gru_model("RNN", weight_ih, weight_hh, bias_ih, bias_hh, seq_len, input_size, hidden_size, return_sequences=False)


def make_simple_rnn_return_sequences_model(input_size, hidden_size, seq_len, seed=None):
    rng = np.random.default_rng(seed)
    weight_ih, weight_hh, bias_ih, bias_hh = _random_rnn_weights(input_size, hidden_size, rng)
    return _build_rnn_or_gru_model("RNN", weight_ih, weight_hh, bias_ih, bias_hh, seq_len, input_size, hidden_size, return_sequences=True)


def make_simple_gru_only_model(input_size, hidden_size, seq_len, seed=None):
    rng = np.random.default_rng(seed)
    weight_ih, weight_hh, bias_ih, bias_hh = _random_gru_weights(input_size, hidden_size, rng)
    return _build_rnn_or_gru_model("GRU", weight_ih, weight_hh, bias_ih, bias_hh, seq_len, input_size, hidden_size, return_sequences=False)


def make_simple_gru_return_sequences_model(input_size, hidden_size, seq_len, seed=None):
    rng = np.random.default_rng(seed)
    weight_ih, weight_hh, bias_ih, bias_hh = _random_gru_weights(input_size, hidden_size, rng)
    return _build_rnn_or_gru_model("GRU", weight_ih, weight_hh, bias_ih, bias_hh, seq_len, input_size, hidden_size, return_sequences=True)

In [10]:
def export_model(name: str, model: Union[nn.Module, onnx.ModelProto], input_shape):
    tmp_model_path = "temporary_model.onnx"

    # input_shape is one of:
    #   int              -> flat feature count (dense-only models); batch
    #                        prepended as dim 0: (1, features)
    #   (C, H, W) tuple   -> conv models; batch prepended as dim 0:
    #                        (1, C, H, W)
    #   (seq_len, F) tuple -> RNN/GRU models; batch goes in the *middle*,
    #                        matching nn.RNN/nn.GRU's batch_first=False (and
    #                        the ONNX RNN/GRU op's own) layout: (seq_len, 1, F)
    if isinstance(input_shape, tuple) and len(input_shape) == 3:
        dummy_input_data = torch.randn(1, *input_shape)
    elif isinstance(input_shape, tuple) and len(input_shape) == 2:
        seq_len, feature_size = input_shape
        dummy_input_data = torch.randn(seq_len, 1, feature_size)
    else:
        dummy_input_data = torch.randn(1, input_shape)

    if isinstance(model, nn.Module):
        # Avoids the "exporting a model while it is in training mode"
        # warning; doesn't change behavior here since none of these models
        # use dropout/batchnorm, but it's the right default for exported
        # fixtures.
        model.eval()
        torch_onnx.export(model, dummy_input_data, tmp_model_path, export_params=True, opset_version=17, dynamo=False)
        with torch.no_grad():
            output = model(dummy_input_data).numpy()
    else:
        # Already an ONNX model (e.g. hand-built via onnx.helper for ops
        # torch.onnx.export doesn't cover yet) - save it as-is and run it
        # through onnxruntime to get the reference output instead of a
        # PyTorch forward pass. From here on both paths are identical.
        onnx.save(model, tmp_model_path)
        input_name = model.graph.input[0].name
        sess = ort.InferenceSession(tmp_model_path)
        output = sess.run(None, {input_name: dummy_input_data.numpy()})[0]

    output_model = export_onnx(tmp_model_path)
    model_output = {
        "model": output_model,
        # Tensor::data on the Rust side is always flat (row-major) regardless
        # of TensorShape, so flatten both input and output fully here rather
        # than relying on tolist()[0], which would leave singleton/batch dims
        # in for conv and RNN/GRU shapes.
        "test_input": dummy_input_data.flatten().tolist(),
        "test_output": output.flatten().tolist(),
    }

    with open(os.path.join(fixtures_path, name), "w") as f:
        print(f"Exporting model: {name}")
        json.dump(model_output, f, indent=2)

In [11]:
fixtures = [
    {
        "name": "dense_simple_model.json",
        "model": SimpleLinearModel(10, 5, 1),
        "input_shape": 10,
    },
    {
        "name": "dense_long_model.json",
        "model": SimpleLinearModel(10, 5, 20),
        "input_shape": 10,
    },
    {
        "name": "dense_large_model.json",
        "model": SimpleLinearModel(100, 100, 5),
        "input_shape": 100,
    },
    {
        "name": "activation_all_types_model.json",
        "model": ActivationModel(10, 5),
        "input_shape": 10,
    },
    {
        "name": "conv_simple_model.json",
        "model": SimpleConvOnlyModel(),
        "input_shape": (1, 4, 4),
    },
    {
        "name": "conv_flatten_model.json",
        "model": ConvFlattenModel(),
        "input_shape": (1, 4, 4),
    },
    {
        "name": "conv_flatten_dense_activation_model.json",
        "model": FullConvModel(5),
        "input_shape": (1, 4, 4),
    },
    {
        "name": "rnn_simple_model.json",
        "model": make_simple_rnn_only_model(input_size=3, hidden_size=5, seq_len=4),
        "input_shape": (4, 3),  # (seq_len, input_size)
    },
    {
        "name": "rnn_return_sequences_model.json",
        "model": make_simple_rnn_return_sequences_model(input_size=3, hidden_size=5, seq_len=4),
        "input_shape": (4, 3),
    },
    {
        "name": "gru_simple_model.json",
        "model": make_simple_gru_only_model(input_size=3, hidden_size=5, seq_len=4),
        "input_shape": (4, 3),
    },
    {
        "name": "gru_return_sequences_model.json",
        "model": make_simple_gru_return_sequences_model(input_size=3, hidden_size=5, seq_len=4),
        "input_shape": (4, 3),
    },
]

In [12]:
for fixture in fixtures:
    export_model(fixture["name"], fixture["model"], fixture["input_shape"])

Exporting model: dense_simple_model.json
Exporting model: dense_long_model.json
Exporting model: dense_large_model.json
Exporting model: activation_all_types_model.json
Exporting model: conv_simple_model.json
Exporting model: conv_flatten_model.json
Exporting model: conv_flatten_dense_activation_model.json
Exporting model: rnn_simple_model.json
Exporting model: rnn_return_sequences_model.json
Exporting model: gru_simple_model.json
Exporting model: gru_return_sequences_model.json
